# Pipeline walkthrough

This notebook runs the pipeline stage by stage and shows the results. Reviewers read this first.

Rules: reuse the code the DAG uses (import it), show evidence after each stage, keep the outputs when you commit.

Replace every *TODO* below. Add cells freely.

In [ ]:
import os, sys
sys.path.insert(0, "/opt/airflow")  # so `ingestion` and `dags` import the same way Airflow sees them

import psycopg2

def query(sql, params=None):
    with psycopg2.connect(
        host=os.environ["WAREHOUSE_HOST"], port=os.environ["WAREHOUSE_PORT"],
        dbname=os.environ["WAREHOUSE_DB"], user=os.environ["WAREHOUSE_USER"], password=os.environ["WAREHOUSE_PASSWORD"],
    ) as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchall() if cur.description else None

query("select version()")

## 1. Extract

*TODO: what do we pull, for which logical date, and how does the client handle failures?*

In [ ]:
# TODO: from ingestion.weather_api import fetch_daily
# logical_date = "2026-08-01"
# rows = fetch_daily(logical_date)
# len(rows), rows[:2]

## 2. Load

*TODO: which table, what is the idempotency mechanism, why that one?*

In [ ]:
# TODO: from ingestion.load import load_daily
# load_daily(logical_date, rows)
# query("select count(*) from raw.weather_daily where date = %s", (logical_date,))

### Re-run safety

Load the same date again and show the count does not change.

In [ ]:
# TODO: load_daily(logical_date, rows)
# query("select count(*) from raw.weather_daily where date = %s", (logical_date,))

## 3. Transform (dbt)

*TODO: what do the staging model and the mart do? Which tests protect what?*

In [ ]:
import subprocess

def dbt(*args):
    r = subprocess.run(["dbt", *args], cwd="/opt/airflow/dbt", capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode

# TODO: dbt("run"); dbt("test")

## 4. Orchestration

*TODO: describe the DAG (tasks, schedule, how the logical date flows into extract/load, retries). Optionally trigger it from here and show its state.*

In [ ]:
# Optional: trigger the DAG for a date and show the run state, e.g.
# subprocess.run(["airflow", "dags", "backfill", "-s", "2026-08-01", "-e", "2026-08-03", "weather_daily"])

## 5. Result

A query on the mart that a business user would recognise.

In [ ]:
# TODO: query("select * from marts.fct_city_daily order by date desc, city limit 10")

## 6. What I would change at scale

*TODO: two or three honest paragraphs. This is discussed in the interview.*